In [ ]:
import osmium
import dagster as dg
import geopandas as gpd

from typing import cast
from shapely import wkt
from shapely.geometry.base import BaseGeometry
from shapely.ops import polygonize
from models.models import FileRef
from osmium import filter, osm, geom

path = "../germany-latest.osm.pbf"

fp = osmium.FileProcessor(path) \
    .with_locations() \
    .with_filter(filter.EntityFilter(osm.WAY)) \
    .with_filter(filter.TagFilter(("boundary", "administrative"))) \
    .with_filter(filter.TagFilter(("admin_level", "2")))

fab = geom.WKTFactory()
lines: list[BaseGeometry] = []

for way in fp:
    way = cast(osm.Way, way)
    lines.append(wkt.loads(fab.create_linestring(way)))

polygons: list[BaseGeometry] = list(polygonize(lines))
gdf = gpd.GeoDataFrame(geometry=polygons, crs=4326).explode()

In [35]:
# Load the pre-built land polygons (global or regional)
land = gpd.read_file("../land-polygons-split-4326/land_polygons.shp")

# Clip land polygons to the country boundary
denmark_land = gpd.clip(land, gdf)

# Dissolve into a single geometry
denmark_land = denmark_land.dissolve().explode()

In [36]:
denmark_land.explore()

In [37]:
gdf.explore()